# HAR V2 — Neural Network Training (PyTorch)

Trains a small neural network on the **same** labeled session data and the **same** 39 IMU features as the V1 random-forest notebook — only the model changes. Evaluates with the identical **leave-one-session-out (LOSO)** protocol and prints an honest **forest-vs-neural-net comparison**, so you can see whether the neural net is actually worth deploying on your dataset.

**Reuses from V1, unchanged:** session folders, `windows_labeled_final.csv`, feature extraction, LOSO split.
**New here:** a PyTorch MLP, and an ExecuTorch export cell for on-device use.

> On small tabular datasets (~1000s of windows), a neural net often **matches or loses to** the forest. That's an expected, honest result — this notebook is designed to *measure* it, not assume the NN wins.


In [ ]:
#1 — config
SESSIONS_DIR = "/content/drive/MyDrive/HAR_data/sessions"   # same folder as V1
LABEL_COLUMN = "Label"
DROP_TRANSITION_WINDOWS = True
MIN_SESSIONS_PER_CLASS = 2
RANDOM_STATE = 42

EPOCHS = 80
BATCH_SIZE = 64
LR = 1e-3
HIDDEN = [128, 64]     # MLP hidden layer sizes

import numpy as np, torch, random
np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE); random.seed(RANDOM_STATE)

In [ ]:
#2 — mount drive + deps
from google.colab import drive
drive.mount('/content/drive')
!pip -q install torch scikit-learn
print("ready")

## Step 1 — Feature extraction (identical to V1)
Same 39 features: per-wrist time-domain stats, magnitude stats, dominant frequency + peak power, and the cross-wrist correlation. Copied verbatim so the NN sees exactly what the forest saw.

In [ ]:
#3 — feature functions (same as V1 training notebook)
import numpy as np, pandas as pd
AXES = ["AccX","AccY","AccZ","GyroX","GyroY","GyroZ"]

def dominant_freq(sig, ts_ms):
    if len(sig) < 8: return 0.0, 0.0
    t = (ts_ms - ts_ms[0]) / 1000.0
    fs = len(t) / max(t[-1], 1e-6)
    x = sig - sig.mean()
    spec = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    freqs = np.fft.rfftfreq(len(x), d=1.0/fs)
    band = (freqs >= 0.8) & (freqs <= 5.0)
    if not band.any(): return 0.0, 0.0
    i = np.argmax(spec[band])
    return float(freqs[band][i]), float(spec[band][i] / (spec[1:].sum() + 1e-9))

def device_features(w, tag):
    f = {}
    for ax in AXES:
        v = w[ax].values
        f[f"{tag}_{ax}_mean"] = v.mean(); f[f"{tag}_{ax}_std"] = v.std()
    acc  = np.sqrt(w.AccX**2 + w.AccY**2 + w.AccZ**2).values
    gyro = np.sqrt(w.GyroX**2 + w.GyroY**2 + w.GyroZ**2).values
    f[f"{tag}_accmag_mean"]  = acc.mean(); f[f"{tag}_accmag_std"] = acc.std()
    f[f"{tag}_accmag_range"] = acc.max() - acc.min()
    f[f"{tag}_gyromag_mean"] = gyro.mean(); f[f"{tag}_gyromag_std"] = gyro.std()
    df_, pp = dominant_freq(acc, w.Timestamp_ms.values)
    f[f"{tag}_dom_freq"] = df_; f[f"{tag}_peak_power"] = pp
    return f, acc

def window_features(streams, w_start, w_end):
    feats, mags = {}, {}
    for tag, s in streams.items():
        w = s[(s.Timestamp_ms >= w_start) & (s.Timestamp_ms < w_end)]
        if len(w) < 8: return None
        fd, acc = device_features(w, tag); feats.update(fd); mags[tag] = acc
    tags = sorted(mags)
    if len(tags) == 2:
        a, b = mags[tags[0]], mags[tags[1]]; n = min(len(a), len(b))
        if n >= 8 and a[:n].std() > 1e-9 and b[:n].std() > 1e-9:
            feats["xdev_accmag_corr"] = float(np.corrcoef(a[:n], b[:n])[0,1])
        else:
            feats["xdev_accmag_corr"] = 0.0
    return feats

In [ ]:
#4 — load sessions (same as V1)
import os, glob
def device_tag(name):
    n = str(name).lower()
    if "left" in n: return "L"
    if "right" in n: return "R"
    return "X"

rows = []
for sdir in sorted(d for d in glob.glob(os.path.join(SESSIONS_DIR, "*")) if os.path.isdir(d)):
    sid = os.path.basename(sdir)
    sens = next((p for n in ["sensors.csv","synced_data.csv"]
                 if os.path.exists(p := os.path.join(sdir, n))), None)
    lab = os.path.join(sdir, "windows_labeled_final.csv")
    if sens is None or not os.path.exists(lab): continue
    s = pd.read_csv(sens); s["Device_Name"] = s["Device_Name"].astype(str).str.strip()
    streams = {device_tag(d): g.reset_index(drop=True) for d, g in s.groupby("Device_Name")}
    for _, r in pd.read_csv(lab).iterrows():
        f = window_features(streams, r.Start_ms, r.End_ms)
        if f is None: continue
        f["session"] = sid; f["y"] = r[LABEL_COLUMN]; rows.append(f)

data = pd.DataFrame(rows).reset_index(drop=True)
# transition flag
data["is_transition"] = False
for sid, g in data.groupby("session"):
    lab = g["y"].values; ch = np.zeros(len(g), bool)
    ch[1:] |= lab[1:] != lab[:-1]; ch[:-1] |= lab[1:] != lab[:-1]
    data.loc[g.index, "is_transition"] = ch
if DROP_TRANSITION_WINDOWS:
    data = data[~data.is_transition].reset_index(drop=True)

per_class = data.groupby("y")["session"].nunique()
weak = per_class[per_class < MIN_SESSIONS_PER_CLASS]
if len(weak):
    print("Dropping under-sampled classes:", list(weak.index))
    data = data[~data.y.isin(weak.index)].reset_index(drop=True)

feature_cols = [c for c in data.columns if c not in ("session","y","is_transition")]
data[feature_cols] = data[feature_cols].fillna(data[feature_cols].median())
print(f"{len(data)} windows, {len(feature_cols)} features, "
      f"{data.y.nunique()} classes, {data.session.nunique()} sessions")
print(data.groupby("y")["session"].agg(windows="count", sessions="nunique"))

## Step 2 — The neural network
A small MLP: input (39 features) → hidden layers with ReLU + dropout → output (one score per class). Features are standardized (mean 0, std 1) per training fold — neural nets need this; forests didn't.

In [ ]:
#5 — model + train/eval helpers
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class MLP(nn.Module):
    def __init__(self, n_in, n_out, hidden):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers += [nn.Linear(prev, n_out)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def train_mlp(Xtr, ytr, n_classes, n_features):
    model = MLP(n_features, n_classes, HIDDEN)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    # class weights for imbalance
    counts = np.bincount(ytr, minlength=n_classes).astype(float)
    w = torch.tensor((counts.sum() / (counts + 1e-9)), dtype=torch.float32)
    lossf = nn.CrossEntropyLoss(weight=w)
    ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                       torch.tensor(ytr, dtype=torch.long))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    model.train()
    for _ in range(EPOCHS):
        for xb, yb in dl:
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
    return model

@torch.no_grad()
def predict_mlp(model, X):
    model.eval()
    return model(torch.tensor(X, dtype=torch.float32)).argmax(1).numpy()

## Step 3 — LOSO evaluation: neural net vs. forest
Both models are trained and tested on the exact same held-out-session folds, so the comparison is fair. Standardization statistics are computed on the training fold only (no leakage).

In [ ]:
#6 — head-to-head LOSO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

X = data[feature_cols].values
le = LabelEncoder(); y = le.fit_transform(data["y"].values)
groups = data["session"].values
n_classes = len(le.classes_)

logo = LeaveOneGroupOut()
rf_true, rf_pred, nn_true, nn_pred = [], [], [], []

for tr, te in logo.split(X, y, groups):
    # --- forest (no scaling needed) ---
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X[tr], y[tr]); rf_pred.extend(rf.predict(X[te])); rf_true.extend(y[te])
    # --- neural net (scale on train fold only) ---
    sc = StandardScaler().fit(X[tr])
    model = train_mlp(sc.transform(X[tr]), y[tr], n_classes, X.shape[1])
    nn_pred.extend(predict_mlp(model, sc.transform(X[te]))); nn_true.extend(y[te])
    print(f"held out {groups[te][0]:30s} "
          f"RF={accuracy_score(y[te], rf.predict(X[te])):.1%}  "
          f"NN={accuracy_score(y[te], predict_mlp(model, sc.transform(X[te]))):.1%}")

rf_acc = accuracy_score(rf_true, rf_pred); nn_acc = accuracy_score(nn_true, nn_pred)
rf_f1  = f1_score(rf_true, rf_pred, average="macro", zero_division=0)
nn_f1  = f1_score(nn_true, nn_pred, average="macro", zero_division=0)

print("\n================ COMPARISON (LOSO) ================")
print(f"{'':20s}{'accuracy':>12s}{'macro-F1':>12s}")
print(f"{'Random Forest':20s}{rf_acc:>11.1%}{rf_f1:>12.3f}")
print(f"{'Neural Net (MLP)':20s}{nn_acc:>11.1%}{nn_f1:>12.3f}")
winner = "Neural Net" if nn_f1 > rf_f1 else "Random Forest"
print(f"\nBetter macro-F1: {winner}")
print("\n--- Neural Net per-class ---")
print(classification_report(nn_true, nn_pred, target_names=le.classes_, zero_division=0))

In [ ]:
#7 — neural net confusion matrix
import matplotlib.pyplot as plt
classes = list(le.classes_)
cm = confusion_matrix(nn_true, nn_pred, labels=range(len(classes)))
cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(1.1*len(classes)+3, 1.0*len(classes)+2))
im = ax.imshow(cmn, cmap="Purples", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes)), classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, f"{cm[i,j]}", ha="center", va="center", fontsize=9,
                color="white" if cmn[i,j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Neural Net — LOSO confusion matrix")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## Step 4 — Train final model on all data + export
Trains the MLP on every session (deploy-ready) and exports to **ExecuTorch** (`.pte`) for on-device inference. Also saves the standardization stats and class order — the phone must apply the same scaling and know the label order.

> If the ExecuTorch export cell fails (the toolchain moves fast), the PyTorch `state_dict` is still saved, and the `.pte` step can be run separately. The comparison results above don't depend on the export.

In [ ]:
#8 — final model + save
sc_final = StandardScaler().fit(X)
final = train_mlp(sc_final.transform(X), y, n_classes, X.shape[1])

import json, os
os.makedirs("/content/drive/MyDrive/HAR_data", exist_ok=True)
torch.save(final.state_dict(), "/content/drive/MyDrive/HAR_data/har_mlp.pt")
meta = {
    "classes": le.classes_.tolist(),
    "feature_names": feature_cols,
    "scaler_mean": sc_final.mean_.tolist(),
    "scaler_std":  sc_final.scale_.tolist(),
    "hidden": HIDDEN,
}
with open("/content/drive/MyDrive/HAR_data/har_mlp_meta.json", "w") as f:
    json.dump(meta, f)
print("saved har_mlp.pt + har_mlp_meta.json")

In [ ]:
#9 — export to ExecuTorch (.pte)
# ExecuTorch's API changes between versions; if this errors, check the current
# torch.export -> executorch flow. The saved .pt + meta above are the fallback.
try:
    !pip -q install executorch
    import torch
    from torch.export import export
    from executorch.exir import to_edge

    example = torch.randn(1, X.shape[1])
    exported = export(final.eval(), (example,))
    edge = to_edge(exported)
    prog = edge.to_executorch()
    with open("/content/drive/MyDrive/HAR_data/har_mlp.pte", "wb") as f:
        f.write(prog.buffer)
    print("exported har_mlp.pte —", len(prog.buffer), "bytes")
except Exception as e:
    print("ExecuTorch export not completed in this environment:", e)
    print("Fallback: use har_mlp.pt + har_mlp_meta.json; run export separately.")

## Reading the results
- **If RF ≥ NN** (likely on a small dataset): that's a legitimate finding — report it. It means the forest remains the better baseline until more (multi-subject) data is collected. The NN path is still built and documented for the future.
- **If NN > RF:** the neural net is worth deploying — proceed to the ExecuTorch-on-Flutter integration.
- **Either way:** this notebook demonstrates the PyTorch/ExecuTorch pipeline end-to-end, which is the V2 deliverable regardless of which model wins today.